# 第 1 章：多臂老虎机 —— RL 的最简形式

> **核心矛盾**：你面对 $K$ 台老虎机，每台的期望奖励 $q_*(a)$ 未知。
> 怎么在 $T$ 步内拿到尽可能多的奖励？

## 学习目标

1. 理解 **探索 vs 利用**（exploration vs exploitation）的两难
2. 掌握 **ε-greedy** 和 **UCB1** 两种经典算法
3. 能用数值实验比较不同策略的 **regret** 曲线
4. 学会用 **递推式更新**（constant-α）处理非平稳问题
5. 知道为什么 **Thompson Sampling** 在随机性问题上近乎最优

In [ ]:
# 常规设置：把项目根加入 sys.path，载入常用库
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'rlenvs').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from utils import set_seed, plot_training_curve, plot_regret_curve, make_interactive
from rlenvs import MultiArmedBandit

set_seed(42)
np.set_printoptions(precision=3)

## 1.1 问题陈述

**$K$-臂老虎机** 是最简单的 RL 问题：
- 有 $K$ 个动作 $a \in \{1, 2, \dots, K\}$，每个动作有一个**固定的真实期望奖励** $q_*(a)$
- 在时刻 $t$，agent 选一个动作 $A_t$，环境返回一个奖励 $R_t \sim \text{某分布，均值 } q_*(A_t)$
- 没有状态、没有时间关联。**唯一的难点是 $q_*$ 未知。**

我们用 Gaussian 版本：$R_t | A_t = a \sim \mathcal{N}(q_*(a), 1)$。

### 一个玩具实例

下面我们建一个 10 臂老虎机，真实均值从 $\mathcal{N}(0, 1)$ 采出，最优臂的均值约 1.5。

In [ ]:
env = MultiArmedBandit(n_arms=10, reward_dist='gaussian', seed=42)
print(f"真实均值 q_*: {env.q_star}")
print(f"最优臂: a* = {env.optimal_arm()}, q_*(a*) = {env.q_star[env.optimal_arm()]:.3f}")

# 画每个臂的奖励分布
fig, ax = plt.subplots(figsize=(8, 4))
for i, q in enumerate(env.q_star):
    samples = np.random.default_rng(0).normal(q, 1, 200)
    ax.scatter(np.full(200, i) + np.random.default_rng(i).uniform(-0.15, 0.15, 200),
               samples, s=8, alpha=0.4)
    ax.plot([i-0.3, i+0.3], [q, q], 'k-', linewidth=2)
ax.set_xlabel('action a')
ax.set_ylabel('reward')
ax.set_title('10-armed bandit: 每臂的真实分布')
plt.show()

注意第 6、第 8 号臂的均值最高。如果 **事先知道** $q_*$，每步都选第 8 号就行。但 RL 的难处正在于 $q_*$ **要靠试错估计**。

## 1.2 估计 $Q(a)$ 的两种方法

我们用 $Q_t(a)$ 表示 $t$ 时刻对 $q_*(a)$ 的估计。两种主流做法：

### 方法 A：样本平均（Sample Average）

$$
Q_t(a) = \frac{\sum_{i=1}^{t-1} R_i \cdot \mathbb{1}[A_i = a]}{N_t(a)}
$$

其中 $N_t(a)$ 是前 $t-1$ 步里选 $a$ 的次数。

**收敛性**：当 $N_t(a) \to \infty$ 时，$Q_t(a) \to q_*(a)$（大数定律）。

### 方法 B：常量 $\alpha$ 递推

$$
Q_{t+1}(a) = Q_t(a) + \alpha \big[ R_t - Q_t(a) \big]
$$

这其实是**指数加权移动平均**——把所有历史奖励按 $\alpha(1-\alpha)^k$ 衰减。

<details>
<summary><b>📝 推导：递推公式 = 指数加权（点开看）</b></summary>

把 $Q_{t+1}$ 展开两步：

$$
Q_{t+1} = (1-\alpha) Q_t + \alpha R_t = (1-\alpha)[(1-\alpha)Q_{t-1} + \alpha R_{t-1}] + \alpha R_t
$$

继续展开，对 $a$ 第 $k$ 次被选（设当时奖励为 $R_k$）：

$$
Q_{k+1} = (1-\alpha)^k Q_1 + \sum_{i=1}^{k} \alpha (1-\alpha)^{k-i} R_i
$$

权重和为 $\sum_{i=1}^{k} \alpha (1-\alpha)^{k-i} = 1 - (1-\alpha)^k \to 1$。
所以越近的奖励权重越大、越远越小，呈指数衰减。
</details>

### 用样本平均在线性时间复杂度内实现

朴素地每次重算 $Q_t$ 是 $O(t)$ 的，但**递推式** 让它变 $O(1)$：

$$
Q_{t+1}(a) = Q_t(a) + \frac{1}{N_t(a)+1} \big[ R_t - Q_t(a) \big]
$$

这等价于样本平均，证明留作思考。

In [ ]:
# 实现：估计 Q 的通用骨架
class BanditAgent:
    """一个抽象的 bandit 智能体：子类只需重写 select_action()."""
    def __init__(self, n_arms, alpha='1/N', init=0.0):
        self.n_arms = n_arms
        self.alpha = alpha  # '1/N' 表示样本平均；float 表示常量 alpha
        self.init = init
        self.Q = np.full(n_arms, init, dtype=float)
        self.N = np.zeros(n_arms, dtype=int)

    def reset(self):
        self.Q[:] = self.init
        self.N[:] = 0

    def select_action(self):
        raise NotImplementedError

    def update(self, action, reward):
        self.N[action] += 1
        alpha = 1.0 / self.N[action] if self.alpha == '1/N' else self.alpha
        self.Q[action] += alpha * (reward - self.Q[action])

# 一个永远"贪心"的 agent（只利用、不探索）
class GreedyAgent(BanditAgent):
    def select_action(self):
        return int(np.argmax(self.Q))


# 跑一个纯贪心 agent，看它能在 1000 步内拿到多少奖励
def run_episode(agent, env, n_steps):
    rewards = []
    opt_actions = []
    a_star = env.optimal_arm()
    env.reset()
    agent.Q[:] = 0
    agent.N[:] = 0
    for t in range(n_steps):
        a = agent.select_action()
        r = env.pull(a)
        agent.update(a, r)
        rewards.append(r)
        opt_actions.append(a == a_star)
    return np.array(rewards), np.array(opt_actions)


env = MultiArmedBandit(n_arms=10, reward_dist='gaussian', seed=42)
greedy = GreedyAgent(10)
rewards, opt = run_episode(greedy, env, n_steps=1000)
print(f"纯贪心：1000 步平均奖励 {rewards.mean():.3f}，选中最优臂的比例 {opt.mean():.1%}")

### 纯贪心的致命缺陷

纯贪心很容易**锁死**：第一次试出某个臂给出正向奖励，就一直选它、永远不试别的臂。

正确做法是引入 **探索**。下面我们实现最经典的探索策略：**$\epsilon$-greedy**。

## 1.3 $\epsilon$-greedy：最经典的探索

策略：

$$
A_t = \begin{cases}
\arg\max_a Q_t(a) & \text{概率 } 1 - \epsilon \\
\text{随机一个臂} & \text{概率 } \epsilon
\end{cases}
$$

- 大 $\epsilon$ → 探索多、收敛快、但天花板低
- 小 $\epsilon$ → 探索少、收敛慢、但天花板高

In [ ]:
class EpsilonGreedyAgent(BanditAgent):
    def __init__(self, n_arms, epsilon=0.1, alpha='1/N'):
        super().__init__(n_arms, alpha=alpha)
        self.epsilon = epsilon

    def select_action(self):
        if np.random.random() < self.epsilon:
            return int(np.random.randint(self.n_arms))
        return int(np.argmax(self.Q))


# 在 200 个随机种子上跑 1000 步，比较不同 epsilon
def benchmark(epsilons, n_seeds=200, n_steps=1000):
    results = {}
    for eps in epsilons:
        all_rewards = np.zeros((n_seeds, n_steps))
        all_opt = np.zeros((n_seeds, n_steps), dtype=bool)
        for seed in range(n_seeds):
            env = MultiArmedBandit(n_arms=10, reward_dist='gaussian', seed=seed)
            agent = EpsilonGreedyAgent(10, epsilon=eps)
            rewards, opt = run_episode(agent, env, n_steps)
            all_rewards[seed] = rewards
            all_opt[seed] = opt
        results[eps] = (all_rewards, all_opt)
    return results

results = benchmark([0.0, 0.01, 0.1, 0.3], n_seeds=200, n_steps=1000)

# 画平均奖励曲线
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for eps, (rw, _) in results.items():
    mean_curve = rw.mean(axis=0)
    smoothed = np.convolve(mean_curve, np.ones(50)/50, mode='valid')
    axes[0].plot(smoothed, label=f'ε={eps}')
axes[0].set_xlabel('step')
axes[0].set_ylabel('avg reward (smoothed w=50)')
axes[0].set_title('平均奖励')
axes[0].legend()

for eps, (_, opt) in results.items():
    mean_curve = opt.mean(axis=0)
    smoothed = np.convolve(mean_curve, np.ones(50)/50, mode='valid')
    axes[1].plot(smoothed, label=f'ε={eps}')
axes[1].set_xlabel('step')
axes[1].set_ylabel('% optimal action')
axes[1].set_title('选中最优臂的比例')
axes[1].legend()
plt.tight_layout()
plt.show()

你应该观察到：

- $\epsilon = 0$（纯贪心）：长期来看最差，约 70-75%
- $\epsilon = 0.1$：约 80%，且后期平台高
- $\epsilon = 0.3$：上升快但天花板低
- $\epsilon = 0.01$：上升慢但终值最高

**没有银弹**——最优 $\epsilon$ 取决于 $T$ 和任务方差。

## 1.4 交互式 widget：调 $\epsilon$ 看效果

下面这个交互组件让你实时调节 $\epsilon$，看奖励曲线变化。

In [ ]:
def plot_eps_demo(epsilon=0.1):
    n_seeds, n_steps = 100, 800
    rw = np.zeros((n_seeds, n_steps))
    opt = np.zeros((n_seeds, n_steps), dtype=bool)
    for seed in range(n_seeds):
        env = MultiArmedBandit(n_arms=10, reward_dist='gaussian', seed=seed)
        agent = EpsilonGreedyAgent(10, epsilon=epsilon)
        r, o = run_episode(agent, env, n_steps)
        rw[seed] = r; opt[seed] = o
    fig, ax = plt.subplots(figsize=(8, 4))
    mean_curve = rw.mean(axis=0)
    sm = np.convolve(mean_curve, np.ones(40)/40, mode='valid')
    ax.plot(sm, color='steelblue', linewidth=2, label='smoothed avg reward')
    ax.axhline(env.q_star.max(), color='crimson', linestyle='--', label='最优 q*')
    ax.set_ylim(-0.5, 2.0)
    ax.set_xlabel('step')
    ax.set_ylabel('reward')
    ax.set_title(f'ε-greedy (ε={epsilon:.2f})')
    ax.legend()
    plt.show()

w = make_interactive(plot_eps_demo,
                     params={'epsilon': (0.1, 0.0, 0.5, 0.01)},
                     layout='hbox')

## 1.5 UCB1：基于不确定性的探索

$\epsilon$-greedy 是**盲目探索**——它随机选臂。能不能更"聪明地"探索？

直觉：如果一个臂**很久没被试过**，我们对它的估计就不确定，应该去试一下。

### UCB1 公式

$$
A_t = \arg\max_a \left[ Q_t(a) + c \sqrt{\frac{\ln t}{N_t(a)}} \right]
$$

- 第一项 $Q_t(a)$：当前估计（**利用**）
- 第二项 $c \sqrt{\frac{\ln t}{N_t(a)}}$：**不确定性奖励**（**探索**）。$N_t(a)$ 越小（越没试过），不确定性越大，越值得探索。

### UCB1 的理论保证（regret 上界）

<details>
<summary><b>📝 完整证明：UCB1 的对数 regret 上界（点开看）</b></summary>

**定理**（Auer et al. 2002）：对任何 $T \geq 1$，UCB1 的期望累计 regret 满足：

$$
\mathbb{E}[R_T] \leq 8 \sum_{a: q_*(a) < q_*(a^*)} \frac{\ln T}{\Delta_a} + \left(1 + \frac{\pi^2}{3}\right) \sum_{a \neq a^*} \Delta_a
$$

其中 $\Delta_a = q_*(a^*) - q_*(a)$ 是次优臂的差距。

**证明思路**：

1. **Hoeffding 不等式**：对有界随机变量 $X \in [0, 1]$，$N$ 次独立采样的均值 $\bar{X}$ 偏离 $\mathbb{E}[X]$ 超过 $\epsilon$ 的概率至多 $2e^{-2N\epsilon^2}$。所以：

$$
|Q_t(a) - q_*(a)| \leq \sqrt{\frac{\ln t}{2 N_t(a)}} \quad \text{w.h.p.}
$$

2. **"选错" 次数上界**：对次优臂 $a$，若它在 $t$ 时刻被选且 $Q_t(a) > Q_t(a^*)$，意味着要么 $a$ 的估计偏高、要么 $a^*$ 的估计偏低。两种情况都用 Hoeffding：

$$
\Pr\left[Q_t(a) > Q_t(a^*) \mid q_*(a) < q_*(a^*)\right] \leq t^{-4}
$$

3. **几何级数**：$\sum_{t=1}^{\infty} t^{-4}$ 收敛，所以次优臂被选的"期望错误次数"是 $O(\ln T / \Delta_a)$。

4. **每错一次贡献 $\Delta_a$ 到 regret**，总和就是定理的形式。

完整证明见 Auer, Cesa-Bianchi & Fischer 2002《Finite-time Analysis of the Multiarmed Bandit Problem》。
</details>

**关键信息**：UCB1 的 regret 是 $O(\ln T)$ 的，比 $\epsilon$-greedy 的 $O(\sqrt{T})$ 渐进更好。

In [ ]:
class UCBAgent(BanditAgent):
    def __init__(self, n_arms, c=2.0):
        super().__init__(n_arms, alpha='1/N')
        self.c = c
        self.t = 0

    def select_action(self):
        self.t += 1
        # 如果有臂没试过，优先试
        untried = np.where(self.N == 0)[0]
        if len(untried) > 0:
            return int(untried[0])
        ucb = self.Q + self.c * np.sqrt(np.log(self.t) / self.N)
        return int(np.argmax(ucb))

# 在同样 200 个种子上对比 ε-greedy 和 UCB
def run_with_agent(agent, env, n_steps):
    env.reset()
    agent.Q[:] = 0; agent.N[:] = 0
    agent.t = 0
    rewards = np.zeros(n_steps)
    opt = np.zeros(n_steps, dtype=bool)
    a_star = env.optimal_arm()
    for t in range(n_steps):
        a = agent.select_action()
        r = env.pull(a)
        agent.update(a, r)
        rewards[t] = r
        opt[t] = (a == a_star)
    return rewards, opt


n_seeds, n_steps = 200, 1000
methods = {
    'ε=0.1':  lambda: EpsilonGreedyAgent(10, epsilon=0.1),
    'ε=0.01': lambda: EpsilonGreedyAgent(10, epsilon=0.01),
    'UCB c=2': lambda: UCBAgent(10, c=2.0),
}
all_rw = {k: np.zeros((n_seeds, n_steps)) for k in methods}
all_opt = {k: np.zeros((n_seeds, n_steps), dtype=bool) for k in methods}
for seed in range(n_seeds):
    env = MultiArmedBandit(n_arms=10, reward_dist='gaussian', seed=seed)
    for name, fac in methods.items():
        r, o = run_with_agent(fac(), env, n_steps)
        all_rw[name][seed] = r
        all_opt[name][seed] = o

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name in methods:
    mean_curve = all_rw[name].mean(axis=0)
    sm = np.convolve(mean_curve, np.ones(50)/50, mode='valid')
    axes[0].plot(sm, label=name)
    mean_opt = all_opt[name].mean(axis=0)
    sm_opt = np.convolve(mean_opt, np.ones(50)/50, mode='valid')
    axes[1].plot(sm_opt, label=name)
axes[0].set_title('平均奖励'); axes[0].legend(); axes[0].set_xlabel('step')
axes[1].set_title('最优臂选中比例'); axes[1].legend(); axes[1].set_xlabel('step')
plt.tight_layout(); plt.show()

你应该看到 **UCB 在早期就迅速接近最优**，因为它的探索是"有目的"的，不是盲目随机。

## 1.6 乐观初始化：另一种探索的"奇技淫巧"

一个意外的发现：**把 $Q_1(a)$ 初始化为一个很大的正数**（比如 +5），用纯贪心也能探索。

直觉：每个臂都看起来"特别好"，纯贪心会去试每一个，每次试都会"失望"（$Q$ 下降），直到试过所有臂。

注意：这种方法在**平稳**问题里和 $\epsilon$-greedy 一样好；但在**非平稳**问题里它会**失去探索能力**（所有 $Q$ 都已收敛到合理值后，纯贪心再也不探索了）。

In [ ]:
class OptimisticGreedyAgent(BanditAgent):
    def __init__(self, n_arms, init=5.0, alpha='1/N'):
        super().__init__(n_arms, alpha=alpha, init=init)
        self.Q[:] = init

    def select_action(self):
        return int(np.argmax(self.Q))

# 通用 runner，调用 agent.reset()
def run_optimistic(env, agent, n_steps):
    env.reset()
    agent.reset()
    rewards, opt = np.zeros(n_steps), np.zeros(n_steps, dtype=bool)
    a_star = env.optimal_arm()
    for t in range(n_steps):
        a = agent.select_action()
        r = env.pull(a); agent.update(a, r)
        rewards[t] = r; opt[t] = (a == a_star)
    return rewards, opt

n_seeds, n_steps = 200, 1000
greedy_rw = np.zeros((n_seeds, n_steps))
optim_rw = np.zeros((n_seeds, n_steps))
for seed in range(n_seeds):
    env = MultiArmedBandit(n_arms=10, reward_dist='gaussian', seed=seed)
    r1, _ = run_optimistic(env, GreedyAgent(10, init=0.0), n_steps)
    r2, _ = run_optimistic(env, OptimisticGreedyAgent(10, init=5.0), n_steps)
    greedy_rw[seed] = r1
    optim_rw[seed] = r2

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(np.convolve(greedy_rw.mean(0), np.ones(50)/50, mode='valid'), label='greedy, init=0')
ax.plot(np.convolve(optim_rw.mean(0), np.ones(50)/50, mode='valid'), label='greedy, init=5')
ax.set_xlabel('step'); ax.set_ylabel('avg reward (w=50)')
ax.set_title('Optimistic init 把纯贪心变成探索器'); ax.legend()
plt.tight_layout(); plt.show()

## 1.7 非平稳问题：为什么需要常量 $\alpha$

到目前为止我们假设 $q_*$ 不变。但现实常常是**非平稳**的：用户的口味会变、市场会变、模型部署的环境会变。

我们用 `MultiArmedBandit(non_stationary=True)` 让 $q_*$ 每步加一个高斯噪声漂移。

**为什么样本平均会失败？** 因为它对所有历史一视同仁，过去 1000 步的奖励和最近的 1 步一样重要。

**为什么常量 $\alpha$ 行？** 因为它的权重指数衰减——最近的奖励最重要。

In [ ]:
n_seeds, n_steps = 1000, 2000
results = {}
for name, alpha in [('sample avg', '1/N'), ('α=0.1', 0.1)]:
    all_rw = np.zeros((n_seeds, n_steps))
    for seed in range(n_seeds):
        env = MultiArmedBandit(n_arms=10, reward_dist='gaussian',
                               non_stationary=True, drift_std=0.01, seed=seed)
        agent = EpsilonGreedyAgent(10, epsilon=0.1, alpha=alpha)
        r, _ = run_with_agent(agent, env, n_steps)
        all_rw[seed] = r
    results[name] = all_rw

fig, ax = plt.subplots(figsize=(8, 4))
for name, rw in results.items():
    ax.plot(np.convolve(rw.mean(0), np.ones(100)/100, mode='valid'), label=name, linewidth=2)
ax.set_xlabel('step'); ax.set_ylabel('avg reward (w=100)')
ax.set_title('非平稳问题：常量 α 完胜样本平均'); ax.legend()
plt.tight_layout(); plt.show()

## 1.8 小结

| 方法 | 探索机制 | regret | 非平稳友好 |
|---|---|---|---|
| 纯贪心 | 无 | $O(T)$ | ✗（锁死） |
| $\epsilon$-greedy | 随机扰动 | $O(\sqrt{T})$ | ✓（常量 $\alpha$） |
| UCB1 | 不确定性 | $O(\ln T)$ | 需改造 |
| 乐观初始化 | 假装每个臂都很好 | 同 $\epsilon$-greedy | ✗（初期有效） |

**核心收获**：

1. RL 的本质是 **在探索和利用之间找平衡**
2. **递推更新** $Q \leftarrow Q + \alpha(R - Q)$ 是 RL 中反复出现的母题
3. **算法不能光看理论曲线**，还要考虑方差、非平稳、超参敏感性

## 1.9 📝 练习

### 练习 1（必做）：实现 Thompson Sampling

Thompson Sampling 的核心思想：对每个臂 $a$ 维护一个**后验分布** $p(q_a | \text{history})$，每步从这个后验采样 $\hat{q}_a$，选 $\arg\max_a \hat{q}_a$。

对 Bernoulli 奖励（每臂真实概率 $\theta_a \in [0, 1]$，给奖励 0/1），用 **Beta 分布**作为共轭先验：

- 先验：$\text{Beta}(1, 1)$（均匀分布）
- 观测 $r=1$ 后：$\alpha \leftarrow \alpha + 1$
- 观测 $r=0$ 后：$\beta \leftarrow \beta + 1$
- 采样：$\hat{\theta}_a \sim \text{Beta}(\alpha_a, \beta_a)$

**任务**：实现 `ThompsonBernoulliAgent`，在 200 个 Bernoulli 10-臂 bandit 上对比它和 UCB1。

> 参考答案：`solutions/ch01_thompson_sampling.ipynb`

---

> 📖 学完本章，先做 `STUDY_GUIDE.md` 里 Ch01 的自测题（4 题），全对再进下一章。

下一章：**第 2 章 — MDP 与贝尔曼方程**。
我们将引入**状态**和**未来**的概念，把单步的 bandit 推广到序列决策的 MDP。